# 面试问题：GraphCL 怎样通过图增强和 NT-Xent 学到稳定的图级表示？

## 可直接复述的回答主线

1. GraphCL 为同一业务图生成两种保语义增强，把它们当正样本，不同图作为 batch 内负样本。
2. 增强不能随意删空图；应限制 edge drop、feature mask，并保存原图到增强图的 provenance。
3. 编码器可用手写归一化邻接矩阵和 GCN 消息传递，再 mean pool 得到图表示。
4. NT-Xent 对每个 view 拉近配对 view、推远其余 2N-2 个 view，并通过 temperature 控制分布锐度。
5. 评测应展示正负 cosine、InfoNCE loss、逐图最近邻和增强后边数，而不是只看 embedding shape。
6. 生产还需大图采样、增强策略搜索、false negative 处理、跨批负样本、分布漂移和下游线性探针。

后续实验会在同一批输入上依次展示朴素基线、手写核心机制、中间过程、失败修正和生产边界。

## 1. 真实案例与输入预览

案例是六个脱敏电商购物篮图，每个图含 5 个商品节点和 5–6 条同购边，节点特征为价格、电子品、配件、耐用品四维。两种固定增强分别删除一条冗余边和遮蔽一个非关键特征，只用于机制教学。

In [1]:
import math  # 计算余弦相似度汇总。
import torch  # 使用基础张量和自动微分手写 GCN 与对比训练。
torch.manual_seed(101)  # 固定模型初始化和训练轨迹。
graphs = [{"id": "basket-phone", "nodes": ["手机", "手机壳", "充电器", "耳机", "贴膜"], "features": [[0.90, 1, 0, 1], [0.12, 0, 1, 0], [0.20, 1, 1, 1], [0.35, 1, 1, 1], [0.05, 0, 1, 0]], "edges": [(0, 1), (0, 2), (0, 3), (0, 4), (1, 4), (2, 3)]}, {"id": "basket-laptop", "nodes": ["笔记本", "鼠标", "键盘", "扩展坞", "电脑包"], "features": [[0.95, 1, 0, 1], [0.15, 1, 1, 1], [0.22, 1, 1, 1], [0.30, 1, 1, 1], [0.18, 0, 1, 1]], "edges": [(0, 1), (0, 2), (0, 3), (0, 4), (1, 2), (3, 4)]}, {"id": "basket-camera", "nodes": ["相机", "镜头", "三脚架", "存储卡", "相机包"], "features": [[0.88, 1, 0, 1], [0.70, 1, 1, 1], [0.28, 0, 1, 1], [0.10, 1, 1, 0], [0.20, 0, 1, 1]], "edges": [(0, 1), (0, 2), (0, 3), (0, 4), (1, 2), (3, 4)]}, {"id": "basket-kitchen", "nodes": ["炒锅", "锅铲", "砧板", "菜刀", "调料架"], "features": [[0.40, 0, 0, 1], [0.08, 0, 1, 1], [0.12, 0, 0, 1], [0.18, 0, 0, 1], [0.10, 0, 1, 1]], "edges": [(0, 1), (0, 2), (2, 3), (0, 3), (1, 4), (3, 4)]}, {"id": "basket-fitness", "nodes": ["瑜伽垫", "哑铃", "弹力带", "水杯", "运动包"], "features": [[0.20, 0, 0, 1], [0.35, 0, 0, 1], [0.12, 0, 1, 1], [0.08, 0, 1, 1], [0.16, 0, 1, 1]], "edges": [(0, 1), (0, 2), (1, 2), (0, 3), (3, 4), (1, 4)]}, {"id": "basket-office", "nodes": ["办公椅", "书桌", "台灯", "收纳盒", "脚垫"], "features": [[0.55, 0, 0, 1], [0.65, 0, 0, 1], [0.22, 1, 0, 1], [0.10, 0, 1, 1], [0.08, 0, 1, 1]], "edges": [(0, 1), (1, 2), (0, 2), (1, 3), (0, 4), (3, 4)]}]  # 定义六个具有商品语义、节点特征和同购边的图。
for graph in graphs:  # 把 Python 特征转换为可训练浮点张量。
    graph["x"] = torch.tensor(graph["features"], dtype=torch.float32)  # 保存当前图的五乘四节点矩阵。
def make_view(graph, view_index):  # 生成可追溯且不删空图的固定增强。
    kept_edges = graph["edges"][:-1] if view_index == 0 else graph["edges"][1:]  # 两个 view 各删除一条不同冗余边。
    features = graph["x"].clone()  # 复制节点特征避免修改原始图。
    mask_node = view_index % features.shape[0]  # 选择确定性的遮蔽节点。
    mask_feature = 0 if view_index == 0 else 3  # 分别遮蔽价格或耐用品维度。
    features[mask_node, mask_feature] = 0.0  # 只遮蔽一个非关键特征值。
    return {"id": graph["id"], "x": features, "edges": kept_edges, "dropped_edge": graph["edges"][-1] if view_index == 0 else graph["edges"][0], "masked": (mask_node, mask_feature)}  # 返回增强图及 provenance。
views_a = [make_view(graph, 0) for graph in graphs]  # 为六个图生成第一种增强。
views_b = [make_view(graph, 1) for graph in graphs]  # 为六个图生成第二种增强。
print("教学实验输入：六个购物篮图")  # 标记下方为离线同购数据。
for graph, view_a, view_b in zip(graphs, views_a, views_b):  # 逐图展示节点、边和增强记录。
    print(f"{graph['id']}: nodes={graph['nodes']} edges={graph['edges']} | viewA drop={view_a['dropped_edge']} mask={view_a['masked']} | viewB drop={view_b['dropped_edge']} mask={view_b['masked']}")  # 输出当前图和两种增强 provenance。

教学实验输入：六个购物篮图
basket-phone: nodes=['手机', '手机壳', '充电器', '耳机', '贴膜'] edges=[(0, 1), (0, 2), (0, 3), (0, 4), (1, 4), (2, 3)] | viewA drop=(2, 3) mask=(0, 0) | viewB drop=(0, 1) mask=(1, 3)
basket-laptop: nodes=['笔记本', '鼠标', '键盘', '扩展坞', '电脑包'] edges=[(0, 1), (0, 2), (0, 3), (0, 4), (1, 2), (3, 4)] | viewA drop=(3, 4) mask=(0, 0) | viewB drop=(0, 1) mask=(1, 3)
basket-camera: nodes=['相机', '镜头', '三脚架', '存储卡', '相机包'] edges=[(0, 1), (0, 2), (0, 3), (0, 4), (1, 2), (3, 4)] | viewA drop=(3, 4) mask=(0, 0) | viewB drop=(0, 1) mask=(1, 3)
basket-kitchen: nodes=['炒锅', '锅铲', '砧板', '菜刀', '调料架'] edges=[(0, 1), (0, 2), (2, 3), (0, 3), (1, 4), (3, 4)] | viewA drop=(3, 4) mask=(0, 0) | viewB drop=(0, 1) mask=(1, 3)
basket-fitness: nodes=['瑜伽垫', '哑铃', '弹力带', '水杯', '运动包'] edges=[(0, 1), (0, 2), (1, 2), (0, 3), (3, 4), (1, 4)] | viewA drop=(1, 4) mask=(0, 0) | viewB drop=(0, 1) mask=(1, 3)
basket-office: nodes=['办公椅', '书桌', '台灯', '收纳盒', '脚垫'] edges=[(0, 1), (1, 2), (0, 2), (1, 3), (0, 4), (3, 4)] | viewA d

## 2. Baseline / 基线：直接平均原始节点特征

不训练图编码器，只对每个增强 view 的节点特征取均值。电子类购物篮彼此很相似，正样本与业务相近负样本难以拉开。

In [2]:
def cosine(left, right):  # 手写两个向量的余弦相似度。
    denominator = left.norm() * right.norm()  # 计算两个向量范数乘积。
    return float((left @ right / denominator).item()) if denominator.item() > 0.0 else 0.0  # 对零向量安全返回零。
baseline_a = torch.stack([view["x"].mean(dim=0) for view in views_a])  # 对第一组增强执行原始特征 mean pooling。
baseline_b = torch.stack([view["x"].mean(dim=0) for view in views_b])  # 对第二组增强执行原始特征 mean pooling。
baseline_positive = [cosine(baseline_a[index], baseline_b[index]) for index in range(len(graphs))]  # 计算六对同图正样本余弦。
baseline_negative = [cosine(baseline_a[left], baseline_b[right]) for left in range(len(graphs)) for right in range(len(graphs)) if left != right]  # 计算全部跨图负样本余弦。
baseline_gap = sum(baseline_positive) / len(baseline_positive) - sum(baseline_negative) / len(baseline_negative)  # 计算正负平均分离度。
print("Baseline 原始特征相似度")  # 标记下表未经过消息传递和训练。
print("graph            positive_cos  hardest_negative")  # 输出基线结果表头。
for index, graph in enumerate(graphs):  # 逐图查找最高负样本相似度。
    hardest = max(cosine(baseline_a[index], baseline_b[other]) for other in range(len(graphs)) if other != index)  # 计算当前图最难负样本。
    print(f"{graph['id']:<16} {baseline_positive[index]:>12.4f} {hardest:>17.4f}")  # 输出当前图正样本和最难负样本。
print(f"Baseline正负平均gap={baseline_gap:.4f}")  # 展示未训练表示的区分能力。

Baseline 原始特征相似度
graph            positive_cos  hardest_negative
basket-phone           0.9890            0.9819
basket-laptop          0.9843            0.9652
basket-camera          0.9793            0.9906
basket-kitchen         0.9911            0.9622
basket-fitness         0.9931            0.9945
basket-office          0.9856            0.9805
Baseline正负平均gap=0.1092


## 3. 底层实现：归一化邻接、两层 GCN、Projection Head 与 NT-Xent

不使用图学习包。邻接矩阵显式加入 self-loop 并计算 D^-1/2 A D^-1/2；两层消息传递后 mean pool，再经过 projection head 归一化。

In [3]:
def normalized_adjacency(node_count, edges):  # 手写无向图的对称归一化邻接矩阵。
    adjacency = torch.eye(node_count, dtype=torch.float32)  # 用单位矩阵加入每个节点 self-loop。
    for source, target in edges:  # 逐同购边写入双向连接。
        adjacency[source, target] = 1.0  # 写入正向邻接。
        adjacency[target, source] = 1.0  # 写入反向邻接。
    degree = adjacency.sum(dim=1)  # 计算含 self-loop 的节点度。
    inverse_sqrt = degree.pow(-0.5)  # 计算 D^-1/2 对角元素。
    return inverse_sqrt[:, None] * adjacency * inverse_sqrt[None, :]  # 返回对称归一化矩阵。
class GraphEncoder(torch.nn.Module):  # 定义显式 forward 的两层 GCN 图编码器。
    def __init__(self, input_dim=4, hidden_dim=16, projection_dim=10):  # 初始化消息传递和投影参数。
        super().__init__()  # 注册 PyTorch 模块参数。
        self.layer_one = torch.nn.Linear(input_dim, hidden_dim, bias=False)  # 定义第一层节点特征变换。
        self.layer_two = torch.nn.Linear(hidden_dim, hidden_dim, bias=False)  # 定义第二层节点特征变换。
        self.projection_one = torch.nn.Linear(hidden_dim, hidden_dim)  # 定义对比 projection 隐层。
        self.projection_two = torch.nn.Linear(hidden_dim, projection_dim)  # 定义最终对比空间。
    def forward(self, node_features, edges, return_debug=False):  # 对一个变长图执行消息传递和 pooling。
        adjacency = normalized_adjacency(node_features.shape[0], edges)  # 构造当前图归一化邻接。
        hidden_one = torch.relu(adjacency @ self.layer_one(node_features))  # 聚合邻居并执行第一层非线性。
        hidden_two = torch.relu(adjacency @ self.layer_two(hidden_one))  # 执行第二层消息传递。
        graph_embedding = hidden_two.mean(dim=0)  # 对所有节点 mean pool 得到图级表示。
        projection = self.projection_two(torch.relu(self.projection_one(graph_embedding)))  # 映射到对比学习空间。
        normalized = projection / projection.norm().clamp_min(1.0e-8)  # 将表示归一化到单位球。
        if return_debug:  # 检查调用方是否需要中间张量。
            return normalized, {"adjacency": adjacency, "hidden_one": hidden_one, "hidden_two": hidden_two, "graph_embedding": graph_embedding}  # 返回表示和消息传递证据。
        return normalized  # 返回单位图表示。
def ntxent_loss(embeddings_a, embeddings_b, temperature=0.20):  # 手写 2N view 的 NT-Xent 损失。
    joined = torch.cat([embeddings_a, embeddings_b], dim=0)  # 拼接两组 view 表示。
    similarity = joined @ joined.T / temperature  # 计算温度缩放后的两两 cosine logits。
    sample_count = embeddings_a.shape[0]  # 读取原始图数量 N。
    losses = []  # 保存每个 anchor 的负对数概率。
    for anchor in range(2 * sample_count):  # 依次把每个 view 当作 anchor。
        positive = anchor + sample_count if anchor < sample_count else anchor - sample_count  # 找到同一原图的另一 view。
        denominator_mask = torch.arange(2 * sample_count) != anchor  # 排除 anchor 自身相似度。
        denominator = torch.logsumexp(similarity[anchor][denominator_mask], dim=0)  # 对正样本和所有负样本做稳定 logsumexp。
        losses.append(-(similarity[anchor, positive] - denominator))  # 保存当前 anchor 的 InfoNCE 损失。
    return torch.stack(losses).mean(), similarity  # 返回批次平均损失和相似度矩阵。
encoder = GraphEncoder()  # 创建待训练 GraphCL 编码器。
history = []  # 保存真实 backward 的损失和梯度轨迹。
for step in range(220):  # 对固定小批次执行多步对比训练。
    encoder.zero_grad(set_to_none=True)  # 清除上一步参数梯度。
    embeddings_a = torch.stack([encoder(view["x"], view["edges"]) for view in views_a])  # 前向编码六个第一视图。
    embeddings_b = torch.stack([encoder(view["x"], view["edges"]) for view in views_b])  # 前向编码六个第二视图。
    loss, similarity = ntxent_loss(embeddings_a, embeddings_b)  # 计算批内正负对比损失。
    loss.backward()  # 对 GCN 和 projection 参数执行真实反向传播。
    gradient_norm = math.sqrt(sum(float((parameter.grad ** 2).sum().item()) for parameter in encoder.parameters()))  # 汇总全部参数梯度二范数。
    with torch.no_grad():  # 在无梯度上下文手动更新参数。
        for parameter in encoder.parameters():  # 遍历 GCN 和投影参数。
            parameter.add_(parameter.grad, alpha=-0.035)  # 使用固定学习率执行 SGD。
    if step % 55 == 0 or step == 219:  # 每五十五步记录可读训练轨迹。
        history.append({"step": step, "loss": loss.item(), "gradient_norm": gradient_norm})  # 保存损失和梯度。
trained_a = torch.stack([encoder(view["x"], view["edges"]) for view in views_a])  # 取得最终第一视图表示。
trained_b = torch.stack([encoder(view["x"], view["edges"]) for view in views_b])  # 取得最终第二视图表示。
preview_embedding, preview_debug = encoder(views_a[0]["x"], views_a[0]["edges"], return_debug=True)  # 对手机篮子取得消息传递中间量。
print("GraphCL训练轨迹=", history)  # 展示损失下降和非零梯度。
print("basket-phone归一化邻接=\n", torch.round(preview_debug["adjacency"] * 1000) / 1000)  # 展示手写图归一化矩阵。
print("第一层节点表示前两行=\n", torch.round(preview_debug["hidden_one"][:2] * 1000) / 1000)  # 展示邻居消息聚合结果。

GraphCL训练轨迹= [{'step': 0, 'loss': 2.3955023288726807, 'gradient_norm': 0.049153159108999464}, {'step': 55, 'loss': 2.3426451683044434, 'gradient_norm': 0.5413667627276915}, {'step': 110, 'loss': 1.5721473693847656, 'gradient_norm': 0.698828688301981}, {'step': 165, 'loss': 1.283972144126892, 'gradient_norm': 3.3223941156925383}, {'step': 219, 'loss': 1.1141302585601807, 'gradient_norm': 2.442897031723747}]
basket-phone归一化邻接=
 tensor([[0.2000, 0.2580, 0.3160, 0.3160, 0.2580],
        [0.2580, 0.3330, 0.0000, 0.0000, 0.3330],
        [0.3160, 0.0000, 0.5000, 0.0000, 0.0000],
        [0.3160, 0.0000, 0.0000, 0.5000, 0.0000],
        [0.2580, 0.3330, 0.0000, 0.0000, 0.3330]])
第一层节点表示前两行=
 tensor([[0.0000, 0.0440, 0.0000, 0.2790, 0.0000, 0.0000, 0.0000, 0.6140, 0.0000,
         0.4850, 0.7090, 0.5110, 0.0000, 0.0000, 1.0330, 0.8400],
        [0.0000, 0.1750, 0.0000, 0.1560, 0.0000, 0.0000, 0.0000, 0.1970, 0.0000,
         0.2240, 0.2320, 0.1340, 0.0000, 0.0000, 0.4150, 0.3740]],
       grad

## 4. 逐图结果与结果解读

比较训练前后每个图的正样本 cosine 和最难负样本；分离度只针对这六个受控购物篮，不能当作普适下游效果。

In [4]:
trained_positive = [cosine(trained_a[index], trained_b[index]) for index in range(len(graphs))]  # 计算训练后六对正样本余弦。
trained_negative = [cosine(trained_a[left], trained_b[right]) for left in range(len(graphs)) for right in range(len(graphs)) if left != right]  # 计算训练后全部负样本余弦。
trained_gap = sum(trained_positive) / len(trained_positive) - sum(trained_negative) / len(trained_negative)  # 计算训练后正负分离度。
print("graph            baseline_pos/hardneg     GraphCL_pos/hardneg      nearest_view")  # 输出逐图同数据对照表头。
nearest_correct = 0  # 统计第一 view 最近邻是否为同图第二 view。
for index, graph in enumerate(graphs):  # 逐图计算难负样本和最近邻。
    baseline_hard = max(cosine(baseline_a[index], baseline_b[other]) for other in range(len(graphs)) if other != index)  # 读取基线最难负样本。
    trained_hard = max(cosine(trained_a[index], trained_b[other]) for other in range(len(graphs)) if other != index)  # 读取训练后最难负样本。
    nearest = max(range(len(graphs)), key=lambda other: cosine(trained_a[index], trained_b[other]))  # 找到另一 view 中最高相似图。
    nearest_correct += int(nearest == index)  # 累加同图检索正确数。
    print(f"{graph['id']:<16} {baseline_positive[index]:>7.3f}/{baseline_hard:<7.3f}       {trained_positive[index]:>7.3f}/{trained_hard:<7.3f}       {graphs[nearest]['id']}")  # 输出当前图表示质量。
print(f"结果解读：正负平均gap从{baseline_gap:.4f}升到{trained_gap:.4f}，同图跨增强检索={nearest_correct}/{len(graphs)}。")  # 解释 GraphCL 在受控数据上的对比效果。

graph            baseline_pos/hardneg     GraphCL_pos/hardneg      nearest_view
basket-phone       0.989/0.982           1.000/1.000         basket-camera
basket-laptop      0.984/0.965           1.000/0.974         basket-laptop
basket-camera      0.979/0.991           0.998/0.999         basket-phone
basket-kitchen     0.991/0.962           0.950/0.543         basket-kitchen
basket-fitness     0.993/0.994           0.988/0.859         basket-fitness
basket-office      0.986/0.981           0.997/0.820         basket-office
结果解读：正负平均gap从0.1092升到1.0956，同图跨增强检索=4/6。


## 5. 失败案例与修正：增强把图删空并遮蔽全部特征

若增强同时删除全部业务边并把节点特征清零，六个图编码成相同向量，positive/negative 都失去语义。受约束增强至少保留连通骨架与大部分特征。

In [5]:
destroyed_views = [{"id": graph["id"], "x": torch.zeros_like(graph["x"]), "edges": []} for graph in graphs]  # 构造删除全部边和特征的过强增强。
destroyed_embeddings = torch.stack([encoder(view["x"], view["edges"]) for view in destroyed_views])  # 编码六个失去业务语义的图。
destroyed_variance = float(destroyed_embeddings.var(dim=0).mean().item())  # 计算跨图表示方差衡量 collapse。
constrained_variance = float(trained_a.var(dim=0).mean().item())  # 计算受约束增强的跨图表示方差。
destroyed_unique = len({tuple(torch.round(row * 10000).tolist()) for row in destroyed_embeddings})  # 统计破坏增强产生的近似唯一表示数。
constrained_unique = len({tuple(torch.round(row * 10000).tolist()) for row in trained_a})  # 统计受约束增强的近似唯一表示数。
print(f"错误行为：destroyed edges=0/features=0，unique_embeddings={destroyed_unique}，variance={destroyed_variance:.8f}")  # 展示过强增强导致表示坍塌。
print(f"修正行为：每图保留{len(views_a[0]['edges'])}条边且只mask一个值，unique_embeddings={constrained_unique}，variance={constrained_variance:.8f}")  # 展示增强约束恢复图间差异。

错误行为：destroyed edges=0/features=0，unique_embeddings=1，variance=0.00000000
修正行为：每图保留5条边且只mask一个值，unique_embeddings=6，variance=0.10984679


## 6. 生产边界

六个图和固定增强会被模型记忆。生产需随机但可审计的增强、连通性/标签保持检查、跨批负样本、false negative 降权、GraphSAINT/子图采样、混合精度、分布式训练，以及冻结 encoder 后的独立下游线性探针。

In [6]:
graphcl_diagnostics = {"graphs": len(graphs), "nodes_per_graph": len(graphs[0]["nodes"]), "final_loss": history[-1]["loss"], "baseline_gap": baseline_gap, "trained_gap": trained_gap, "nearest_correct": nearest_correct, "destroyed_unique": destroyed_unique, "constrained_unique": constrained_unique}  # 汇总数据、训练、检索和失败指标。
print("生产监控快照：", graphcl_diagnostics)  # 输出 GraphCL 训练应持续观察的信号。

生产监控快照： {'graphs': 6, 'nodes_per_graph': 5, 'final_loss': 1.1141302585601807, 'baseline_gap': 0.10921514232953389, 'trained_gap': 1.0956174763540427, 'nearest_correct': 4, 'destroyed_unique': 1, 'constrained_unique': 6}


## 7. 最小回归测试

断言覆盖图规模、真实梯度、损失下降、表示分离、跨增强检索和坍塌修正。

In [7]:
assert len(graphs) >= 6 and all(len(graph["edges"]) >= 5 for graph in graphs)  # 保证至少六个具有非平凡边的真实业务图。
assert history[-1]["loss"] < history[0]["loss"] and all(row["gradient_norm"] > 0.0 for row in history)  # 保证 GraphCL 实际执行 forward/backward 并学习。
assert trained_gap > baseline_gap  # 保证同数据正负平均分离度得到改善。
assert nearest_correct >= 4  # 保证多数图能跨增强检索回自身，同时保留难负样本混淆这一真实现象。
assert destroyed_unique == 1 and constrained_unique > destroyed_unique  # 保证过强增强坍塌真实复现并被约束增强修正。
assert all(len(view["edges"]) > 0 and torch.count_nonzero(view["x"]) > 0 for view in views_a + views_b)  # 保证正常增强不删空结构或全部特征。